
# Workshop 1 — Source Profiling

**Team 8 — Tsunami Monitoring** · Data Analysis Programming 2026-II

This notebook profiles the four data sources of the project using the raw samples stored in `workshop_1/samples/`.

**Scope:** Pacific basin (HazEL regions 80–89 and the USGS rectangle lat −60..65, lon 100..300), 2010–2025, earthquakes M ≥ 6.

| Role | Source | Format | Samples used |
|---|---|---|---|
| Historical 1 | NOAA/NCEI HazEL tsunami events | JSON (REST) | `noaa_tsunamis_*.json`, `noaa_descriptor_*.json` |
| Historical 2 | USGS FDSN earthquake catalog | CSV | `usgs_earthquakes_*.csv` |
| Incremental 1 | USGS real-time feed (M4.5+, past month) | GeoJSON | `usgs_realtime_*.geojson` |
| Incremental 2 | tsunami.gov warning centers (NTWC, PTWC) | Atom + CAP | `ntwc_*.xml`, `ptwc_*.xml`, `*_cap*.xml` |

Sections: 1 Setup · 2 Load · 3 Profiling · 4 Coordinates, timestamps, units · 5 Data-quality problems · 6 Cross-source overlap · 7 Conclusions


## 1. Setup

In [3]:

import json
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 80)

SAMPLES = Path("../workshop_1/samples")
print("Samples folder:", SAMPLES.resolve())
for path in sorted(SAMPLES.iterdir()):
    if path.name != ".gitkeep":
        print(f"{path.stat().st_size:>10,} bytes  {path.name}")


Samples folder: /home/linmonp/tsunami-monitoring/workshop_1/samples
       604 bytes  noaa_descriptor_causes.json
     1,390 bytes  noaa_descriptor_regions.json
       357 bytes  noaa_descriptor_validity.json
       517 bytes  noaa_descriptor_warning_statuses.json
   102,014 bytes  noaa_runups_event_4682_page1.json
    87,520 bytes  noaa_tsunamis_2010_2025.json
     8,190 bytes  noaa_tsunamis_2011.json
    45,356 bytes  noaa_tsunamis_page2.json
     1,983 bytes  ntwc_20261107_050434.xml
     1,983 bytes  ntwc_20261107_140301.xml
     4,327 bytes  ntwc_cap1_20261107_050434.xml
     4,327 bytes  ntwc_cap1_20261107_140301.xml
     1,967 bytes  ptwc_20261107_050434.xml
     1,967 bytes  ptwc_20261107_140301.xml
     3,932 bytes  ptwc_cap1_20261107_050434.xml
     3,932 bytes  ptwc_cap1_20261107_140301.xml
    34,959 bytes  usgs_earthquakes_2011_m6.csv
   498,562 bytes  usgs_realtime_20261107_050434.geojson
   506,851 bytes  usgs_realtime_20261107_140301.geojson


## 2. Load the samples

### 2.1 NOAA HazEL — tsunami events (historical)

In [4]:

noaa_files = sorted(f for f in SAMPLES.glob("noaa_tsunamis*.json") if "2011" not in f.name)
print("Files:", [f.name for f in noaa_files])

noaa = pd.DataFrame(
    [item for f in noaa_files for item in json.loads(f.read_text(encoding="utf-8"))["items"]]
)
print("Rows read:", len(noaa), "| duplicated ids:", noaa["id"].duplicated().sum())

def load_descriptor(name):
    items = json.loads((SAMPLES / f"noaa_descriptor_{name}.json").read_text(encoding="utf-8"))["items"]
    return {int(i["id"]): i["description"] for i in items}

causes = load_descriptor("causes")
validity = load_descriptor("validity")
warning_statuses = load_descriptor("warning_statuses")
regions = load_descriptor("regions")

noaa["cause"] = noaa["causeCode"].map(causes)
noaa["validity"] = noaa["eventValidity"].map(validity)
noaa["warning_status"] = noaa["warningStatusId"].map(warning_statuses)
noaa["region"] = noaa["regionCode"].map(regions)

# HazEL stores the date in six separate columns: build one UTC timestamp.
# Hour/minute/second can be missing; keep a flag instead of inventing values.
noaa["time_complete"] = noaa["hour"].notna() & noaa["minute"].notna()
noaa["time_utc"] = (
    pd.to_datetime(noaa[["year", "month", "day"]], errors="coerce", utc=True)
    + pd.to_timedelta(noaa["hour"].fillna(0), unit="h")
    + pd.to_timedelta(noaa["minute"].fillna(0), unit="m")
    + pd.to_timedelta(noaa["second"].fillna(0), unit="s")
)
print("Unparseable dates:", noaa["time_utc"].isna().sum())
noaa[["id", "time_utc", "time_complete", "latitude", "longitude", "eqMagnitude", "cause", "validity", "region"]].head()


Files: ['noaa_tsunamis_2010_2025.json', 'noaa_tsunamis_page2.json']
Rows read: 317 | duplicated ids: 0
Unparseable dates: 1


,id,time_utc,time_complete,latitude,longitude,eqMagnitude,cause,validity,region
0,3490,2014-01-29 15:31:00.000+00:00,True,64.495,11.153,NaN,Landslide,Definite Tsunami,Northeast Atlantic Ocean
1,4522,2010-01-03 21:48:05.300+00:00,True,-8.743,157.477,6.6,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
2,4523,2010-01-03 22:36:27.900+00:00,True,-8.799,157.346,7.1,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
3,4542,2010-01-05 12:15:32.200+00:00,True,-9.019,157.551,6.8,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
4,4622,2010-01-12 21:53:10.000+00:00,True,18.457,-72.533,7.0,Earthquake and Landslide,Definite Tsunami,Caribbean Sea


### 2.2 USGS FDSN catalog — earthquakes (historical)

In [5]:

usgs_file = sorted(SAMPLES.glob("usgs_earthquakes_*.csv"))[-1]
print("File:", usgs_file.name)

usgs_hist = pd.read_csv(usgs_file)
usgs_hist["time_utc"] = pd.to_datetime(usgs_hist["time"], utc=True)
usgs_hist["updated_utc"] = pd.to_datetime(usgs_hist["updated"], utc=True)
print("Rows:", len(usgs_hist), "| period:", usgs_hist["time_utc"].min(), "->", usgs_hist["time_utc"].max())
usgs_hist.head()


File: usgs_earthquakes_2011_m6.csv
Rows: 207 | period: 2011-01-01 09:56:58.120000+00:00 -> 2011-12-27 15:21:56.840000+00:00


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,time_utc,updated_utc
0,2011-12-27T15:21:56.840Z,51.842,95.911,15.0,6.6,mww,545,17.6,NaN,0.81,us,usp000jcq2,2022-04-08T20:25:09.055Z,"45 km NNE of Saryg-Sep, Russia",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-27 15:21:56.840+00:00,2022-04-08 20:25:09.055+00:00
1,2011-12-26T04:48:08.870Z,-16.198,-173.862,90.0,6.0,mww,536,35.4,NaN,0.95,us,usp000jcna,2022-04-08T20:25:06.544Z,"27 km SSW of Hihifo, Tonga",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-26 04:48:08.870+00:00,2022-04-08 20:25:06.544+00:00
2,2011-12-15T10:10:07.210Z,-32.718,-179.099,32.0,6.0,mww,423,21.6,NaN,1.09,us,usp000jc7p,2022-04-08T20:24:26.506Z,south of the Kermadec Islands,earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-15 10:10:07.210+00:00,2022-04-08 20:24:26.506+00:00
3,2011-12-14T05:04:58.630Z,-7.551,146.809,135.0,7.1,mww,485,10.6,NaN,0.78,us,usp000jc5z,2022-04-08T20:24:25.265Z,"25 km SSE of Wau, Papua New Guinea",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-14 05:04:58.630+00:00,2022-04-08 20:24:25.265+00:00
4,2011-12-13T07:52:11.930Z,0.041,123.030,161.0,6.0,mww,536,13.9,NaN,0.95,us,usp000jc4z,2022-04-08T20:24:22.823Z,"55 km S of Gorontalo, Indonesia",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-13 07:52:11.930+00:00,2022-04-08 20:24:22.823+00:00


### 2.3 USGS real-time feed — earthquakes (incremental)

In [6]:

rt_files = sorted(SAMPLES.glob("usgs_realtime_*.geojson"))
print("Snapshots available:", [f.name for f in rt_files])
rt_file = rt_files[-1]  # profile the latest snapshot
feed = json.loads(rt_file.read_text(encoding="utf-8"))
print("Feed title:", feed["metadata"]["title"], "| count:", feed["metadata"]["count"])
print("Generated:", pd.to_datetime(feed["metadata"]["generated"], unit="ms", utc=True))

usgs_rt = pd.json_normalize(feed["features"])
usgs_rt.columns = [c.replace("properties.", "") for c in usgs_rt.columns]
# GeoJSON coordinates are [longitude, latitude, depth] (longitude first)
usgs_rt[["lon", "lat", "depth"]] = pd.DataFrame(usgs_rt["geometry.coordinates"].tolist(), index=usgs_rt.index)
usgs_rt = usgs_rt.drop(columns=["geometry.coordinates", "geometry.type", "type"])
usgs_rt["time_utc"] = pd.to_datetime(usgs_rt["time"], unit="ms", utc=True)
usgs_rt["updated_utc"] = pd.to_datetime(usgs_rt["updated"], unit="ms", utc=True)
usgs_rt[["id", "time_utc", "updated_utc", "mag", "magType", "status", "tsunami", "lat", "lon", "depth", "place"]].head()


Snapshots available: ['usgs_realtime_20261107_050434.geojson', 'usgs_realtime_20261107_140301.geojson']
Feed title: USGS Magnitude 4.5+ Earthquakes, Past Month | count: 726
Generated: 2026-11-07 14:03:01+00:00


,id,time_utc,updated_utc,mag,magType,status,tsunami,lat,lon,depth,place
0,us7000twrt,2026-11-07 11:35:38.338+00:00,2026-11-07 11:55:28.040+00:00,4.9,mb,reviewed,0,-18.2483,-12.9445,10.000,southern Mid-Atlantic Ridge
1,us7000twrq,2026-11-07 11:07:43.355+00:00,2026-11-07 13:39:16.859+00:00,5.5,mww,reviewed,0,36.1884,141.4404,31.748,"74 km NE of Hasaki, Japan"
2,us7000twrp,2026-11-07 10:53:12.299+00:00,2026-11-07 12:46:07.040+00:00,4.5,mb,reviewed,0,-21.6011,-68.6413,110.716,"57 km SW of Ollagüe, Chile"
3,us7000twrh,2026-11-07 09:47:33.240+00:00,2026-11-07 12:35:21.040+00:00,4.7,mb,reviewed,0,0.0075,125.7691,38.531,"156 km ESE of Modisi, Indonesia"
4,us7000twqw,2026-11-07 07:14:10.868+00:00,2026-11-07 07:41:39.040+00:00,4.6,mb,reviewed,0,-57.9579,-25.8023,103.682,South Sandwich Islands region


### 2.4 tsunami.gov — Atom feeds and CAP messages (incremental)

In [7]:

NS = {
    "a": "http://www.w3.org/2005/Atom",
    "geo": "http://www.w3.org/2003/01/geo/wgs84_pos#",
    "cap": "urn:oasis:names:tc:emergency:cap:1.2",
}

def parse_cap(path):
    """Flatten a CAP message into one dictionary."""
    root = ET.parse(path).getroot()
    info = root.find("cap:info", NS)
    row = {
        "identifier": root.findtext("cap:identifier", namespaces=NS),
        "sent": root.findtext("cap:sent", namespaces=NS),
        "status": root.findtext("cap:status", namespaces=NS),
        "msgType": root.findtext("cap:msgType", namespaces=NS),
    }
    for tag in ["event", "urgency", "severity", "certainty", "headline"]:
        row[tag] = info.findtext(f"cap:{tag}", namespaces=NS)
    for param in info.findall("cap:parameter", NS):
        row[param.findtext("cap:valueName", namespaces=NS)] = param.findtext("cap:value", namespaces=NS)
    return row

def read_center(center):
    """Read the latest Atom snapshot of a warning center, plus the CAP of each entry."""
    atom_file = sorted(SAMPLES.glob(f"{center}_2*.xml"))[-1]  # excludes *_cap*.xml
    stamp = atom_file.stem.split(f"{center}_")[1]
    root = ET.parse(atom_file).getroot()
    rows = []
    for number, entry in enumerate(root.findall("a:entry", NS), start=1):
        row = {
            "center": center,
            "snapshot": stamp,
            "entry_id": entry.findtext("a:id", namespaces=NS),
            "title": entry.findtext("a:title", namespaces=NS),
            "updated": entry.findtext("a:updated", namespaces=NS),
            "lat": entry.findtext("geo:lat", namespaces=NS),
            "lon": entry.findtext("geo:long", namespaces=NS),
        }
        cap_file = SAMPLES / f"{center}_cap{number}_{stamp}.xml"
        if cap_file.exists():
            row.update({f"cap_{k}": v for k, v in parse_cap(cap_file).items()})
        rows.append(row)
    print(f"{center}: {atom_file.name} -> {len(rows)} entries")
    return pd.DataFrame(rows)

tsunami_gov = pd.concat([read_center("ntwc"), read_center("ptwc")], ignore_index=True)
tsunami_gov.T


ntwc: ntwc_20261107_140301.xml -> 1 entries
ptwc: ptwc_20261107_140301.xml -> 1 entries


,0,1
center,ntwc,ptwc
snapshot,20261107_140301,20261107_140301
entry_id,urn:uuid:6d7fa756-1e86-4fe3-841a-e10e7ebb7430,urn:uuid:ba64bc82-07f7-45fb-ab6c-25420fca6851
title,"75 miles SW of Eureka, California",BONIN ISLANDS JAPAN REGION
updated,2026-11-05T05:01:47Z,2026-10-25T02:27:01Z
lat,40.396,28.
lon,-125.484,139.9
cap_identifier,PAAQ-1-to0j1g,PHEB-1-26297050
cap_sent,2026-11-05T05:01:47-00:00,2026-10-25T02:27:01-00:00
cap_status,Actual,Actual



## 3. Profiling

For every source: row count, duplicated identifiers, null percentage per field, data type and value ranges.
The workshop requires **at least 200 records per source, or the full source if it is smaller**.


In [8]:

def profile(df, name, id_col):
    print(f"{name}: {len(df)} rows x {df.shape[1]} columns")
    print(f"  duplicated {id_col}: {df[id_col].duplicated().sum()}")
    print("  meets the 200-record requirement:", "yes" if len(df) >= 200 else "NO -> widen the sample unless this is the full source")
    table = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
    })
    numeric = df.select_dtypes("number")
    table["min"] = numeric.min()
    table["max"] = numeric.max()
    return table.sort_values("null_%", ascending=False)


### 3.1 NOAA HazEL events

In [9]:
profile(noaa, 'NOAA HazEL tsunami events', 'id')

NOAA HazEL tsunami events: 317 rows x 56 columns
  duplicated id: 0
  meets the 200-record requirement: yes


,dtype,null_count,null_%,n_unique,min,max
tsMtIi,float64,316,99.7,1,8.400,8.400
missing,float64,315,99.4,2,10.000,667.000
missingAmountOrder,float64,315,99.4,2,1.000,3.000
housesDamaged,float64,310,97.8,7,1.000,280920.000
damageMillionsDollars,float64,309,97.5,8,0.200,220136.600
missingTotal,float64,308,97.2,9,1.000,667.000
missingAmountOrderTotal,float64,308,97.2,3,1.000,3.000
housesDamagedAmountOrder,float64,305,96.2,3,1.000,4.000
housesDestroyed,float64,304,95.9,11,1.000,123661.000
injuries,float64,299,94.3,14,1.000,31943.000


### 3.2 USGS FDSN catalog

In [10]:
profile(usgs_hist, 'USGS FDSN catalog', 'id')

USGS FDSN catalog: 207 rows x 24 columns
  duplicated id: 0
  meets the 200-record requirement: yes


,dtype,null_count,null_%,n_unique,min,max
horizontalError,float64,207,100.0,0,NaN,NaN
magError,float64,207,100.0,0,NaN,NaN
dmin,float64,207,100.0,0,NaN,NaN
magNst,float64,173,83.6,32,15.000,299.000
depthError,float64,158,76.3,37,2.400,20.400
rms,float64,3,1.4,71,0.470,1.480
time,str,0,0.0,207,NaN,NaN
latitude,float64,0,0.0,207,-62.496,70.931
gap,float64,0,0.0,169,9.500,169.200
nst,int64,0,0.0,170,40.000,832.000


### 3.3 USGS real-time feed

In [11]:
profile(usgs_rt, 'USGS real-time feed', 'id')

USGS real-time feed: 726 rows x 31 columns
  duplicated id: 0
  meets the 200-record requirement: yes


,dtype,null_count,null_%,n_unique,min,max
tz,object,726,100.0,0,NaN,NaN
alert,str,652,89.8,3,NaN,NaN
mmi,float64,641,88.3,65,0.000000e+00,8.750000e+00
felt,float64,598,82.4,33,0.000000e+00,2.018000e+03
cdi,float64,598,82.4,28,0.000000e+00,7.900000e+00
mag,float64,0,0.0,29,4.500000e+00,7.700000e+00
id,str,0,0.0,726,NaN,NaN
url,str,0,0.0,726,NaN,NaN
updated,int64,0,0.0,725,1.792143e+12,1.794060e+12
time,int64,0,0.0,726,1.791469e+12,1.794051e+12


### 3.4 tsunami.gov

In [12]:

print("Entries per center (each Atom feed only keeps the latest message):")
print(tsunami_gov.groupby("center").size())
profile(tsunami_gov, "tsunami.gov (NTWC + PTWC)", "entry_id")


Entries per center (each Atom feed only keeps the latest message):
center
ntwc    1
ptwc    1
dtype: int64
tsunami.gov (NTWC + PTWC): 2 rows x 26 columns
  duplicated entry_id: 0
  meets the 200-record requirement: NO -> widen the sample unless this is the full source


,dtype,null_count,null_%,n_unique,min,max
cap_WEAK53,str,1,50.0,1,NaN,NaN
cap_ProductDefinition,str,1,50.0,1,NaN,NaN
cap_NWSUGC,str,1,50.0,1,NaN,NaN
entry_id,str,0,0.0,2,NaN,NaN
updated,str,0,0.0,2,NaN,NaN
lat,str,0,0.0,2,NaN,NaN
lon,str,0,0.0,2,NaN,NaN
title,str,0,0.0,2,NaN,NaN
center,str,0,0.0,2,NaN,NaN
snapshot,str,0,0.0,1,NaN,NaN


## 4. Coordinates, timestamps and units

### 4.1 Coordinate validity (WGS84: latitude −90..90, longitude −180..180)

In [13]:

def check_coords(df, lat, lon, name):
    lat_v = pd.to_numeric(df[lat], errors="coerce")
    lon_v = pd.to_numeric(df[lon], errors="coerce")
    return {
        "source": name,
        "rows": len(df),
        "missing": int((lat_v.isna() | lon_v.isna()).sum()),
        "lat_out_of_range": int(((lat_v < -90) | (lat_v > 90)).sum()),
        "lon_out_of_range": int(((lon_v < -180) | (lon_v > 180)).sum()),
        "at_0_0": int(((lat_v == 0) & (lon_v == 0)).sum()),
    }

pd.DataFrame([
    check_coords(noaa, "latitude", "longitude", "NOAA HazEL"),
    check_coords(usgs_hist, "latitude", "longitude", "USGS FDSN"),
    check_coords(usgs_rt, "lat", "lon", "USGS real-time"),
    check_coords(tsunami_gov, "lat", "lon", "tsunami.gov"),
])


,source,rows,missing,lat_out_of_range,lon_out_of_range,at_0_0
0,NOAA HazEL,317,32,0,0,0
1,USGS FDSN,207,0,0,0,0
2,USGS real-time,726,0,0,0,0
3,tsunami.gov,2,0,0,0,0


### 4.2 Timestamp formats and time zones

In [14]:

print("NOAA date columns (year, month, day, hour, minute, second), example:")
print(noaa.loc[noaa["time_complete"], ["year", "month", "day", "hour", "minute", "second"]].head(2).to_string(index=False))
print("USGS CSV time (ISO 8601, UTC):", usgs_hist["time"].iloc[0])
print("USGS real-time time (epoch milliseconds, UTC):", usgs_rt["time"].iloc[0], "->", usgs_rt["time_utc"].iloc[0])
print("tsunami.gov Atom updated (ISO 8601, UTC):", tsunami_gov["updated"].iloc[0])
if "cap_EventOriginTime" in tsunami_gov:
    print("tsunami.gov CAP EventOriginTime:", tsunami_gov["cap_EventOriginTime"].dropna().iloc[0])

ranges = pd.DataFrame({
    "first": [noaa["time_utc"].min(), usgs_hist["time_utc"].min(), usgs_rt["time_utc"].min()],
    "last": [noaa["time_utc"].max(), usgs_hist["time_utc"].max(), usgs_rt["time_utc"].max()],
}, index=["NOAA HazEL", "USGS FDSN", "USGS real-time"])
print("Time coverage of each sample:")
ranges


NOAA date columns (year, month, day, hour, minute, second), example:
 year  month  day  hour  minute  second
 2014      1 29.0  15.0    31.0     NaN
 2010      1  3.0  21.0    48.0     5.3
USGS CSV time (ISO 8601, UTC): 2011-12-27T15:21:56.840Z
USGS real-time time (epoch milliseconds, UTC): 1794051338338 -> 2026-11-07 11:35:38.338000+00:00
tsunami.gov Atom updated (ISO 8601, UTC): 2026-11-05T05:01:47Z
tsunami.gov CAP EventOriginTime: 2026-11-05T04:56:04-00:00
Time coverage of each sample:


,first,last
NOAA HazEL,2010-01-02 00:00:00.000+00:00,2025-12-12 02:44:11.000+00:00
USGS FDSN,2011-01-01 09:56:58.120+00:00,2011-12-27 15:21:56.840+00:00
USGS real-time,2026-10-08 14:11:53.009+00:00,2026-11-07 11:35:38.338+00:00


### 4.3 Units and severity scales

In [15]:

print("USGS FDSN magnitude types:")
print(usgs_hist["magType"].value_counts().to_string())
print("USGS real-time magnitude types:")
print(usgs_rt["magType"].value_counts().to_string())

units = pd.DataFrame([
    ["NOAA HazEL", "eqMagnitude", "magnitude (type not given)", "eqDepth: km (verify in HazEL docs)"],
    ["USGS FDSN / real-time", "mag + magType", "mww, mb, mwc, ms, ml ...", "depth: km"],
    ["tsunami.gov Atom/CAP", "Preliminary Magnitude / EventPreliminaryMagnitude", "Ml, Mw ... (preliminary)", "CAP EventDepth: km; text bulletin: miles"],
], columns=["source", "severity field", "magnitude type", "depth unit"])
units


USGS FDSN magnitude types:
magType
mww    155
mb      33
mwc     12
mwb      6
ms       1
USGS real-time magnitude types:
magType
mb     609
mww    104
mwr      8
mw       2
ml       2
mwb      1


,source,severity field,magnitude type,depth unit
0,NOAA HazEL,eqMagnitude,magnitude (type not given),eqDepth: km (verify in HazEL docs)
1,USGS FDSN / real-time,mag + magType,"mww, mb, mwc, ms, ml ...",depth: km
2,tsunami.gov Atom/CAP,Preliminary Magnitude / EventPreliminaryMagnitude,"Ml, Mw ... (preliminary)",CAP EventDepth: km; text bulletin: miles



## 5. Data-quality problems

Each problem is computed from the samples, with one example record.


In [16]:

problems = []

def add(problem, source, evidence, example):
    problems.append({"problem": problem, "source": source, "evidence": evidence, "example": example})

# 1. NOAA: official warning status missing for most events
missing = noaa["warningStatusId"].isna()
add("Official warning status missing", "NOAA HazEL",
    f"{missing.sum()} of {len(noaa)} events ({missing.mean():.0%}) have no warningStatusId",
    f"event id {noaa.loc[missing, 'id'].iloc[0]}")

# 2. NOAA: doubtful / erroneous events
bad = noaa[noaa["eventValidity"] <= 2]
add("Erroneous or doubtful tsunami events", "NOAA HazEL",
    f"{len(bad)} events with validity <= 2: {bad['validity'].value_counts().to_dict()}",
    f"event id {bad['id'].iloc[0]} -> {bad['validity'].iloc[0]}")

# 3. NOAA: incomplete time of day
no_time = ~noaa["time_complete"]
add("Time of day missing", "NOAA HazEL",
    f"{no_time.sum()} events have no hour/minute (only the date is known)",
    f"event id {noaa.loc[no_time, 'id'].iloc[0]} on {noaa.loc[no_time, 'time_utc'].iloc[0].date()}")

# 4. NOAA: no coordinates
no_coord = noaa["latitude"].isna() | noaa["longitude"].isna()
add("Events without coordinates", "NOAA HazEL",
    f"{no_coord.sum()} events cannot be placed on a map",
    f"event id {noaa.loc[no_coord, 'id'].iloc[0]} ({noaa.loc[no_coord, 'locationName'].iloc[0]})")

# 5. NOAA: tsunamis that were not caused by an earthquake
non_eq = ~noaa["causeCode"].isin([1, 3, 4, 5])
add("Tsunamis without an earthquake cause", "NOAA HazEL",
    f"{non_eq.sum()} events: {noaa.loc[non_eq, 'cause'].value_counts().to_dict()}",
    f"event id {noaa.loc[non_eq, 'id'].iloc[0]} -> {noaa.loc[non_eq, 'cause'].iloc[0]}")

# 6. USGS CSV: columns that are completely empty
empty_cols = [c for c in usgs_hist.columns if usgs_hist[c].isna().all()]
add("Columns that are always empty", "USGS FDSN (CSV)",
    f"{len(empty_cols)} columns are 100% null: {empty_cols}",
    "every row")

# 7. USGS: identifiers with different formats
odd = usgs_hist[usgs_hist["net"] != "us"]
if len(odd):
    add("Identifiers from different networks", "USGS FDSN (CSV)",
        f"net values: {usgs_hist['net'].value_counts().to_dict()}",
        f"id {odd['id'].iloc[0]} (net={odd['net'].iloc[0]}), {odd['place'].iloc[0]}")

# 8. USGS real-time: the tsunami flag does not mean a tsunami
flagged = usgs_rt[usgs_rt["tsunami"] == 1]
add("`tsunami` flag is only a bulletin indicator", "USGS real-time",
    f"{len(flagged)} of {len(usgs_rt)} events have tsunami=1; the flag does not mean a tsunami happened",
    f"id {flagged['id'].iloc[0]} ({flagged['place'].iloc[0]})" if len(flagged) else "none in this snapshot")

# 9. tsunami.gov: location as free text, depth in different units
atom_text = tsunami_gov[["center", "title"]].head(2).to_dict("records")
add("Location as free text; units mixed", "tsunami.gov",
    "region is text ('75 miles SW of ...'); the text bulletin gives depth in miles, the CAP in km",
    str(atom_text[0]))

pd.DataFrame(problems)


,problem,source,evidence,example
0,Official warning status missing,NOAA HazEL,256 of 317 events (81%) have no warningStatusId,event id 3490
1,Erroneous or doubtful tsunami events,NOAA HazEL,"15 events with validity <= 2: {'Questionable Tsunami': 12, 'Event that only ...",event id 5407 -> Questionable Tsunami
2,Time of day missing,NOAA HazEL,55 events have no hour/minute (only the date is known),event id 4762 on 2010-04-13
3,Events without coordinates,NOAA HazEL,32 events cannot be placed on a map,event id 5483 (NORTHWEST ATLANTIC OCEAN)
4,Tsunamis without an earthquake cause,NOAA HazEL,"88 events: {'Landslide': 35, 'Meteorological': 34, 'Volcano': 13, 'Unknown':...",event id 3490 -> Landslide
5,Columns that are always empty,USGS FDSN (CSV),"3 columns are 100% null: ['dmin', 'horizontalError', 'magError']",every row
6,Identifiers from different networks,USGS FDSN (CSV),"net values: {'us': 206, 'official': 1}","id official20110311054624120_30 (net=official), 2011 Great Tohoku Earthquake..."
7,`tsunami` flag is only a bulletin indicator,USGS real-time,6 of 726 events have tsunami=1; the flag does not mean a tsunami happened,"id us6000u4r9 (Bonin Islands, Japan region)"
8,Location as free text; units mixed,tsunami.gov,region is text ('75 miles SW of ...'); the text bulletin gives depth in mile...,"{'center': 'ntwc', 'title': '75 miles SW of Eureka, California'}"



## 6. Cross-source overlap

Matching rule used for this exploration (to be justified in Workshop 3):
two records describe the same event when their origin times differ by at most **15 minutes** and their epicenters are at most **100 km** apart.
When the NOAA record has no time of day, only the **date** must match (flagged as `date_only`).


In [17]:

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

def norm_lon(lon):
    """USGS can return longitudes above 180 in a Pacific query; bring them to -180..180."""
    return ((lon + 180) % 360) - 180

def find_pairs(noaa_df, usgs_df, max_minutes=15, max_km=100):
    rows = []
    for _, n in noaa_df.dropna(subset=["latitude", "longitude"]).iterrows():
        window = pd.Timedelta(minutes=max_minutes) if n["time_complete"] else pd.Timedelta(days=1)
        close = usgs_df[(usgs_df["time_utc"] - n["time_utc"]).abs() <= window]
        best = None
        for _, u in close.iterrows():
            dist = haversine_km(n["latitude"], n["longitude"], u["latitude"], norm_lon(u["longitude"]))
            if dist <= max_km and (best is None or dist < best["distance_km"]):
                best = {
                    "noaa_id": n["id"], "usgs_id": u["id"],
                    "noaa_time": n["time_utc"], "usgs_time": u["time_utc"],
                    "dt_minutes": round((u["time_utc"] - n["time_utc"]).total_seconds() / 60, 1),
                    "distance_km": round(dist, 1),
                    "noaa_mag": n["eqMagnitude"], "usgs_mag": u["mag"], "usgs_magType": u["magType"],
                    "noaa_depth": n["eqDepth"], "usgs_depth": u["depth"],
                    "date_only": not n["time_complete"],
                    "place": u["place"],
                }
        if best:
            rows.append(best)
    return pd.DataFrame(rows)

# Compare only NOAA events inside the period covered by the USGS sample
start = usgs_hist["time_utc"].min().normalize()
end = usgs_hist["time_utc"].max().normalize() + pd.Timedelta(days=1)
noaa_period = noaa[
    (noaa["time_utc"] >= start) & (noaa["time_utc"] <= end) & noaa["causeCode"].isin([1, 3, 4, 5])
]
print(f"Period compared: {start.date()} -> {end.date()}")
print(f"NOAA earthquake-caused tsunamis in that period: {len(noaa_period)}")

pairs = find_pairs(noaa_period, usgs_hist)
print(f"Pairs found (NOAA -> USGS): {len(pairs)}")
pairs


Period compared: 2011-01-01 -> 2011-12-28
NOAA earthquake-caused tsunamis in that period: 16
Pairs found (NOAA -> USGS): 16


,noaa_id,usgs_id,noaa_time,usgs_time,dt_minutes,distance_km,noaa_mag,usgs_mag,usgs_magType,noaa_depth,usgs_depth,date_only,place
0,5406,usp000huc2,2011-02-11 20:05:30.700+00:00,2011-02-11 20:05:30.910+00:00,0.0,15.9,6.8,6.9,mww,28.0,26.0,False,"21 km N of Tomé, Chile"
1,5407,usp000huvq,2011-02-21 23:51:42.300+00:00,2011-02-21 23:51:42.350+00:00,0.0,0.0,6.1,6.1,mww,6.0,5.9,False,"6 km SE of Christchurch, New Zealand"
2,5412,usp000hvhj,2011-03-09 02:45:20.000+00:00,2011-03-09 02:45:20.330+00:00,0.0,0.0,7.3,7.3,mww,32.0,32.0,False,"120 km SE of ?funato, Japan"
3,5413,official20110311054624120_30,2011-03-11 05:46:24.100+00:00,2011-03-11 05:46:24.120+00:00,0.0,0.0,9.1,9.1,mww,29.0,29.0,False,"2011 Great Tohoku Earthquake, Japan"
4,5419,usp000hst4,2011-01-09 10:03:44.200+00:00,2011-01-09 10:03:43.990+00:00,-0.0,0.2,6.5,6.5,mww,24.0,22.0,False,"110 km WNW of Isangel, Vanuatu"
5,5429,usp000hzf6,2011-04-07 14:32:43.200+00:00,2011-04-07 14:32:43.290+00:00,0.0,0.0,7.1,7.1,mww,42.0,42.0,False,"29 km ESE of Ishinomaki, Japan"
6,5430,usp000j48h,2011-07-06 19:03:18.200+00:00,2011-07-06 19:03:18.260+00:00,0.0,0.0,7.6,7.6,mww,17.0,17.0,False,Kermadec Islands region
7,5431,usp000j4gp,2011-07-10 00:57:10.800+00:00,2011-07-10 00:57:10.800+00:00,0.0,0.0,7.0,7.0,mww,23.0,23.0,False,"177 km ESE of Ishinomaki, Japan"
8,5432,usp000j3mq,2011-06-24 03:09:39.400+00:00,2011-06-24 03:09:39.470+00:00,0.0,0.0,7.3,7.3,mww,52.0,52.0,False,"Fox Islands, Aleutian Islands, Alaska"
9,5436,usp000j6rj,2011-08-20 18:19:23.500+00:00,2011-08-20 18:19:23.550+00:00,0.0,0.0,7.0,7.1,mww,28.0,28.0,False,"64 km S of Port-Vila, Vanuatu"


In [18]:

if len(pairs):
    summary = pd.DataFrame({
        "pairs": [len(pairs)],
        "median |dt| (min)": [pairs["dt_minutes"].abs().median()],
        "median distance (km)": [pairs["distance_km"].median()],
        "median |mag diff|": [(pairs["noaa_mag"] - pairs["usgs_mag"]).abs().median()],
    })
    display(summary)
    print("Pairs with the largest differences:")
    display(
        pairs.assign(mag_diff=(pairs["noaa_mag"] - pairs["usgs_mag"]).abs())
        .sort_values(["mag_diff", "distance_km"], ascending=False)
        .head(5)[["noaa_id", "usgs_id", "noaa_mag", "usgs_mag", "usgs_magType",
                  "noaa_depth", "usgs_depth", "distance_km", "dt_minutes"]]
    )


,pairs,median |dt| (min),median distance (km),median |mag diff|
0,16,0.0,0.0,0.0


Pairs with the largest differences:


,noaa_id,usgs_id,noaa_mag,usgs_mag,usgs_magType,noaa_depth,usgs_depth,distance_km,dt_minutes
14,5931,usp000hvkh,6.3,6.5,mwc,15.0,15.0,0.0,0.0
0,5406,usp000huc2,6.8,6.9,mww,28.0,26.0,15.9,0.0
10,5437,usp000j6r4,7.1,7.2,mww,32.0,32.0,0.0,0.0
11,5438,usp000j7dc,6.8,6.9,mww,32.0,32.0,0.0,0.0
9,5436,usp000j6rj,7.0,7.1,mww,28.0,28.0,0.0,0.0


### 6.1 tsunami.gov bulletin vs USGS real-time feed

In [19]:

caps = tsunami_gov.dropna(subset=["cap_EventOriginTime"]) if "cap_EventOriginTime" in tsunami_gov else pd.DataFrame()
rows = []
for _, t in caps.iterrows():
    t_time = pd.to_datetime(t["cap_EventOriginTime"], utc=True)
    t_lat, t_lon = (float(x) for x in t["cap_EventLatLon"].split()[0].split(","))
    close = usgs_rt[(usgs_rt["time_utc"] - t_time).abs() <= pd.Timedelta(minutes=15)]
    for _, u in close.iterrows():
        dist = haversine_km(t_lat, t_lon, u["lat"], u["lon"])
        if dist <= 100:
            rows.append({
                "center": t["center"], "bulletin": t["cap_identifier"], "usgs_id": u["id"],
                "bulletin_time": t_time, "usgs_time": u["time_utc"],
                "dt_seconds": round((u["time_utc"] - t_time).total_seconds(), 1),
                "distance_km": round(dist, 1),
                "bulletin_mag": f"{t['cap_EventPreliminaryMagnitude']} {t['cap_EventPreliminaryMagnitudeType']}",
                "usgs_mag": f"{u['mag']} {u['magType']}",
                "bulletin_depth": t["cap_EventDepth"], "usgs_depth_km": round(u["depth"], 1),
                "usgs_tsunami_flag": u["tsunami"],
                "bulletin_says": t["cap_severity"] + " / " + t["cap_certainty"],
            })
bulletin_pairs = pd.DataFrame(rows)
print("Bulletin/USGS pairs found:", len(bulletin_pairs), "(a message can fall outside the 30-day window of the feed)")
bulletin_pairs.T


Bulletin/USGS pairs found: 1 (a message can fall outside the 30-day window of the feed)


,0
center,ntwc
bulletin,PAAQ-1-to0j1g
usgs_id,nc75462637
bulletin_time,2026-11-05 04:56:04+00:00
usgs_time,2026-11-05 04:56:02.670000+00:00
dt_seconds,-1.3
distance_km,8.7
bulletin_mag,4.5 Ml
usgs_mag,4.96 mw
bulletin_depth,13 kilometers



## 7. Conclusions for the canonical mapping

Fill in with the team after reviewing the outputs above.

| Topic | Finding | Decision for the data contract |
|---|---|---|
| Timestamps | NOAA splits the date in 6 columns; USGS uses ISO or epoch ms | Store `timestamp` in UTC; keep a `time_precision` flag |
| Coordinates | USGS GeoJSON is [lon, lat, depth]; Pacific queries return lon > 180 | Normalize to WGS84 −180..180 |
| Severity | Magnitude type differs across sources (Ml, mw, mww, mb) | Store the value and its type |
| Duplicates | Identifiers are source-specific | Event key = (source, id); cross-source links go in `event_group` |
| Missing values | `warningStatusId` and time of day are often missing | Keep NULL, never impute; document per field |
| Change detection | USGS revises `updated`/`mag`; tsunami.gov keeps only the latest message | Watermark on USGS `updated`; poll tsunami.gov often |
